[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO_2026/blob/main/Orsenigo_0_information_theory_solutions.ipynb)

# Lesson 0 — Information theory: a more general perspective on the brain (solutions)

**Computational Neuroscience 2026/27 · Davide Orsenigo**

**Wednesday 4 November 2026, room G**

This notebook goes with the slides of the lesson. We work on one resting-state fMRI recording and ask a
single question in three steps: how uncertain is the signal of one brain region, how much does one region
tell about another, and can three regions share something that no pair of them shows? The definitions are
those of Cover and Thomas, *Elements of Information Theory* (2nd ed., Wiley 2006), chapter 2.

| Part | Slides | Content |
|---|---|---|
| 1 | Part I | From BOLD signals to symbols; the entropy of one region |
| 2 | Part I | Two regions: joint and conditional entropy |
| 3 | Part II | Mutual information; zero correlation is not independence; the bias of counting |
| 4 | Part II | Mutual information and functional connectivity in 19,900 pairs of regions |
| 5 | Part III | Three signals: copies, XOR and triplets of brain regions |

Everything runs on the default Colab runtime (numpy, scipy, matplotlib): no installation and no Google Drive
needed. Every cell runs in a few seconds. Information is measured in **bits**: all logarithms are in base 2,
unless a formula says $\ln$.

This version contains the code of every exercise (after the word **Solution**) and an answer to every
question. It runs from top to bottom without errors; the numbers it prints are the "Expected" values of the
exercises.

---
# Setup

The data are one resting-state fMRI run of one participant of the Human Connectome Project (HCP; Van Essen et
al., *NeuroImage* 80:62, 2013): 1200 **volumes** (brain images), one every 0.72 s, 14.4 minutes at rest. In each
volume the BOLD signal is averaged over each of the 200 cortical **regions** of the Schaefer atlas (Schaefer et
al., *Cereb Cortex* 28:3095, 2018): 200 signals of 1200 values. The regions come in the order of seven
resting-state networks (Yeo et al., 2011), first the left hemisphere (LH, regions 0–99), then the right one
(RH, 100–199).

The setup cell downloads the file (1.8 MB) from the course repository and **z-scores** each region: it
subtracts the mean of the signal and divides by its standard deviation, so that every region has mean 0 and
standard deviation 1. The address names one commit of the repository, and the cell checks the file against
its SHA-256, so every run uses the same data. HCP data are distributed under the WU-Minn HCP Open Access Data
Use Terms.

In [ ]:
import hashlib
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy, norm

# Figure style: minimal axes, Arial-like font, palette and colormaps of figkit (D. Orsenigo)
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
PALETTE = ["#3B6E8F", "#E07A5F", "#81B29A", "#E0A458", "#8D6A9F", "#C65B6B", "#5B7080", "#3D3D3D"]
FC_CMAP = LinearSegmentedColormap.from_list(   # diverging petrol-brick, white at 0
    "fk_div", ["#2C5F86", "#6FA0BE", "#BFD3DF", "#F3F3F1", "#EFC0B2", "#D77C6A", "#B23A48"])
MI_CMAP = LinearSegmentedColormap.from_list(   # sequential cream-plum, for quantities >= 0
    "fk_warm", ["#FBEEE6", "#F6C19A", "#EE8A5C", "#D8503F", "#A4255B", "#5E1246"])
STATE_CMAP = ListedColormap(["#2C5F86", "#BFD3DF", "#EFC0B2", "#B23A48"])   # the four states 0..3
plt.rcParams.update({
    "figure.dpi": 110, "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10, "axes.titlesize": 10.5, "axes.labelsize": 10, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "mathtext.fontset": "dejavusans",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8, "axes.grid": False,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.prop_cycle": plt.cycler(color=PALETTE),
    "xtick.direction": "out", "ytick.direction": "out", "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8, "lines.linewidth": 1.4, "lines.markersize": 4.5,
    "legend.frameon": False, "image.cmap": "magma"})

# Download the data (1.8 MB) into the working directory, if they are not there yet
DATA_URL = "https://raw.githubusercontent.com/alanturin-g/Computational_Neuroscience_UNITO_2026/b4d76249bd82eba9518c0a5b2f8a1f53561d59ae/Datasets/HCP_BOLD.npz"
if not os.path.exists("HCP_BOLD.npz"):
    urllib.request.urlretrieve(DATA_URL, "HCP_BOLD.npz")
if hashlib.sha256(open("HCP_BOLD.npz", "rb").read()).hexdigest() != "c8b3676ff01434390e9412f34be4d7be513af903493f287b656c96a905f195da":
    print("Warning: HCP_BOLD.npz is not the file this notebook was written for")
bold = np.load("HCP_BOLD.npz")["signal"]                       # shape (200 regions, 1200 volumes)
bold = (bold - bold.mean(axis=1, keepdims=True)) / bold.std(axis=1, keepdims=True)   # z-score each region
N, T = bold.shape
TR = 0.72                                                     # seconds between two volumes

# The seven networks of Yeo et al. (2011) in the order of the Schaefer atlas (200 regions, 7 networks):
# regions per network, left hemisphere (regions 0-99), then right hemisphere (regions 100-199)
NETWORKS = ["Visual", "Somatomotor", "Dorsal attention", "Ventral attention", "Limbic", "Control", "Default"]
network = np.concatenate([np.repeat(np.arange(7), [14, 16, 13, 12, 6, 13, 26]),
                          np.repeat(np.arange(7), [15, 19, 13, 11, 6, 17, 19])])


def region_name(i):
    return f"{i} ({'RH' if i >= 100 else 'LH'} {NETWORKS[network[i]]})"


print(f"{N} regions, {T} volumes ({T * TR / 60:.1f} minutes)")

---
# Part 1 — From BOLD signals to symbols

We want the **probabilities of the states** of a signal, and we estimate them by counting, without assuming
their shape. A continuous signal never takes exactly the same value twice, so we cannot count how often a value
occurs. We cut the range of each z-scored signal into bins instead, and count how often the signal falls in
each bin. We use four bins, with edges at −1, 0 and +1 standard deviations: the four **states** 0, 1, 2, 3. The
edges are symmetric around the mean, and every state holds more than 10% of the volumes in every region. More
bins would keep more detail, but would need more samples to be counted well (Part 3 shows the cost). For a
Gaussian signal the four probabilities would be 0.16, 0.34, 0.34 and 0.16.

The probability of a state is estimated by the fraction of the 1200 volumes in that state. Calling a fraction
of time a probability assumes that the statistics of the signal do not change during the 14 minutes
(*stationarity*). Cutting throws some detail away, because two values in the same bin become the same symbol;
Part 4 measures how much.

In [ ]:
X = np.digitize(bold, [-1, 0, 1])                    # states 0, 1, 2, 3, same shape as bold
n_show = 300                                         # the first 300 volumes (3.6 minutes)
t_min = np.arange(n_show) * TR / 60

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1.25, 1]})
for i in range(5):
    axes[0].plot(t_min, bold[i, :n_show] - 5 * i, lw=0.8)
axes[0].set_yticks([-5 * i for i in range(5)], [f"region {i}" for i in range(5)])
axes[0].set_xlabel("time (min)")
axes[0].set_title("BOLD, z-scored")
im = axes[1].imshow(X[:5, :n_show], aspect="auto", cmap=STATE_CMAP, interpolation="nearest",
                    extent=[0, t_min[-1], 4.5, -0.5])
axes[1].set_yticks(range(5), [f"region {i}" for i in range(5)])
axes[1].set_xlabel("time (min)")
axes[1].set_title("four states")
cb = plt.colorbar(im, ax=axes[1], ticks=[0.375, 1.125, 1.875, 2.625], shrink=0.85)
cb.ax.set_yticklabels(["0: below −1", "1: −1 to 0", "2: 0 to 1", "3: above 1"])
plt.tight_layout()
plt.show()
print("probability of the four states, all regions together:", (np.bincount(X.ravel()) / X.size).round(3))
print("least visited state, region by region: at least", (np.stack([np.bincount(x, minlength=4) for x in X]) / T).min().round(3))

## Exercise 1: The entropy of one region

The **surprise** of a state of probability $p$ is $-\log_2 p$: zero for a certain state, one bit for one
side of a fair coin, larger for rarer states. The **entropy** of a signal $X$ is its average surprise, each
state $x$ weighted by how often it occurs:

$$H(X) = -\sum_x p(x)\,\log_2 p(x), \qquad p(x) = \frac{n(x)}{\sum_{x'} n(x')},$$

where $n(x)$ is the number of volumes in state $x$. A state that never occurs does not contribute, because
$p \log p \to 0$ for $p \to 0$. Write the function `entropy_bits`, which takes the array of counts and returns
$H$ in bits. Normalise by the sum of the array, as in the formula: the same function then works when it is
given probabilities instead of counts. Then run the next cell, which counts the states of region 0.

*Expected:* $H(X_0) \approx 1.866$ bits. With four states the largest possible value is $\log_2 4 = 2$ bits,
reached only if the four states are equally likely; a Gaussian signal cut at the same edges gives 1.90 bits.
`scipy.stats.entropy` gives the same value with `base=2`; without it, scipy uses natural logarithms and
returns nats (1 nat = 1.44 bits).

**Solution**

In [ ]:
def entropy_bits(counts):
    """Entropy in bits of the distribution given by counts (an array of non-negative numbers)."""
    counts = np.asarray(counts, dtype=float)
    p = counts[counts > 0] / counts.sum()
    return -np.sum(p * np.log2(p))

In [ ]:
counts0 = np.bincount(X[0], minlength=4)
print("counts of the four states in region 0:", counts0)
print(f"H(X0) = {entropy_bits(counts0):.3f} bits;  scipy: {scipy_entropy(counts0, base=2):.3f} bits "
      f"({scipy_entropy(counts0):.3f} nats without base=2)")
p_gauss = np.diff(norm.cdf([-np.inf, -1, 0, 1, np.inf]))      # a Gaussian cut at -1, 0, +1
print(f"Gaussian signal cut at the same edges: {entropy_bits(p_gauss):.3f} bits; four equal states: 2 bits")

---
# Part 2 — Two regions: joint and conditional entropy

Take two regions, $X$ and $Y$. At each time point the pair is in one of $4 \times 4 = 16$ **joint states**
$(x, y)$, and $p(x, y)$ is the fraction of time points in which $X = x$ and $Y = y$. Summing $p(x, y)$ over $y$
gives back $p(x)$, the *marginal* probability of $X$. Two signals are **independent** when
$p(x, y) = p(x)\,p(y)$ for every pair of states. The **joint entropy** is the entropy of the pair:

$$H(X, Y) = -\sum_{x,y} p(x,y)\,\log_2 p(x,y).$$

The function below counts the joint states. `np.vstack` puts the signals one above the other, so that each
column is one time point, and `np.unique(..., axis=1)` counts the distinct columns.

In [ ]:
def joint_entropy(*signals):
    """Entropy in bits of the joint state of one or more discrete signals of the same length."""
    states = np.vstack(signals)                                 # one row per signal, one column per time point
    _, counts = np.unique(states, axis=1, return_counts=True)   # how often each joint state occurs
    return entropy_bits(counts)


P01 = np.zeros((4, 4))
np.add.at(P01, (X[0], X[1]), 1 / T)                    # p(x0, x1): one count of 1/T per time point
print("joint probabilities of regions 0 (rows) and 1 (columns):")
print(P01.round(3))
print("row sums, p(x0):", P01.sum(axis=1).round(3))
print(f"H(X0) = {joint_entropy(X[0]):.3f},  H(X1) = {joint_entropy(X[1]):.3f},  "
      f"H(X0, X1) = {joint_entropy(X[0], X[1]):.3f} bits")

If the two regions were independent, every joint probability would be a product, the surprises of the two
states would add, and $H(X_0, X_1)$ would be $1.866 + 1.876 = 3.742$ bits. The pair has 3.640 bits: 0.102 bits
less, because the regions depend on each other.

If we know that $Y = y$, the probabilities of $X$ become $p(x|y) = p(x,y)/p(y)$: among the time points with
$Y = y$, the fraction with $X = x$. Their entropy, $H(X|Y{=}y) = -\sum_x p(x|y)\log_2 p(x|y)$, averaged over the
states of $Y$, is the **conditional entropy** $H(X|Y)$: the uncertainty left on $X$ when $Y$ is known. Since
$p(x,y) = p(x|y)\,p(y)$, the logarithm of a product splits into a sum, and the average gives the **chain
rule**

$$H(X, Y) = H(Y) + H(X|Y), \qquad\text{that is}\qquad H(X|Y) = H(X, Y) - H(Y).$$

## Exercise 2: Conditional entropy

Use the chain rule to compute the two conditional entropies of regions 0 and 1.

*Expected:* $H(X_0|X_1) \approx 1.764$ bits and $H(X_1|X_0) \approx 1.774$ bits. Knowing one region removes
0.102 bits of uncertainty about the other, in both directions.

**Solution**

In [ ]:
H0, H1, H01 = joint_entropy(X[0]), joint_entropy(X[1]), joint_entropy(X[0], X[1])
H0_given_1 = H01 - H1
H1_given_0 = H01 - H0

print(f"H(X0 | X1) = {H0_given_1:.3f} bits,  H(X1 | X0) = {H1_given_0:.3f} bits")
print(f"uncertainty removed: {H0 - H0_given_1:.3f} and {H1 - H1_given_0:.3f} bits")

**Question 1.** Before running the next cell: can the joint entropy of regions 0–3 be smaller than that of
regions 0 and 1 (3.640 bits)?

**Answer.** No. Adding a region can only split each joint state of the first two into several, never merge them, so
$H(X_0, X_1, X_2, X_3) \ge H(X_0, X_1)$; formally, $H(X_0,X_1,X_2,X_3) = H(X_0,X_1) + H(X_2,X_3|X_0,X_1)$ and a
conditional entropy is never negative. The next cell gives 6.43 bits, less than the sum of the four entropies
because the regions depend on each other.

In [ ]:
H_0123 = joint_entropy(X[0], X[1], X[2], X[3])
print(f"H(X0, X1, X2, X3) = {H_0123:.3f} bits; the four entropies add up to "
      f"{sum(joint_entropy(X[i]) for i in range(4)):.3f} bits")
print(f"joint states seen: {np.unique(X[:4], axis=1).shape[1]} of 4^4 = 256, in {T} samples")

Four regions already have 256 joint states, and 1200 volumes visit only 135 of them: many probabilities are
estimated from one or two counts. With ten regions there would be about a million joint states. Part 3 shows
what counting with too few samples does; in Part 5 we will estimate quantities of three regions from their
correlations instead.

---
# Part 3 — Mutual information

The **mutual information** is the uncertainty on $X$ removed by knowing $Y$. With the chain rule it becomes a
symmetric expression:

$$I(X;Y) = H(X) - H(X|Y) = H(X) + H(Y) - H(X,Y).$$

Writing the three entropies as averages over the joint states gives a third form,

$$I(X;Y) = \sum_{x,y} p(x,y)\,\log_2\frac{p(x,y)}{p(x)\,p(y)}.$$

The ratio inside the logarithm compares what we observe with what independence would give. $I$ is never
negative, and it is zero only for independent signals (slides, Part II).

## Exercise 3: Mutual information

Write `mutual_information` with the form $H(X) + H(Y) - H(X,Y)$, using `joint_entropy`.

*Expected:* regions 4 and 5 (correlation 0.80) share $I \approx 0.562$ bits; regions 0 and 1 (correlation
0.35) share 0.102 bits, the uncertainty removed in Exercise 2.

**Solution**

In [ ]:
def mutual_information(x, y):
    """Mutual information in bits between two discrete signals."""
    return joint_entropy(x) + joint_entropy(y) - joint_entropy(x, y)

In [ ]:
for i, j in [(4, 5), (0, 1)]:
    rho = np.corrcoef(bold[i], bold[j])[0, 1]
    print(f"regions {i} and {j}: correlation {rho:.2f},  I = {mutual_information(X[i], X[j]):.3f} bits")

## Zero correlation is not independence

The Pearson correlation is the average product of the two z-scored signals: it measures how well a straight
line fits the cloud of points. The next cell builds three pairs of signals: a line with noise, a parabola and a
circle. Each has 20,000 samples, independent of each other, and each signal is cut into 8 equally populated
bins before computing $I$. With so many samples we can afford more bins than with 1200 volumes: the bias of
Exercise 4 is only $(8-1)^2/(2 \cdot 20000 \ln 2) = 0.002$ bits. Equally populated bins (quantiles) adapt to the
very different distributions of $x$ and $y$; the parabola's $y$, for example, is never much below 0.

In [ ]:
def quantile_states(x, n_states=8):
    """Cut x into n_states equally populated bins: states 0 .. n_states-1."""
    return np.digitize(x, np.quantile(x, np.linspace(0, 1, n_states + 1)[1:-1]))


rng = np.random.default_rng(1)
n = 20_000
u, e = rng.standard_normal((2, n))
angle = rng.uniform(0, 2 * np.pi, n)
pairs = {
    "line": (u, 0.8 * u + 0.6 * e),                   # a Gaussian pair with correlation 0.8
    "parabola": (u, u**2 + 0.3 * e),
    "circle": (np.cos(angle) + 0.05 * u, np.sin(angle) + 0.05 * e),
}
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
for ax, (name, (a, b)) in zip(axes, pairs.items()):
    rho = np.corrcoef(a, b)[0, 1]
    info = mutual_information(quantile_states(a), quantile_states(b))
    ax.plot(a[:2000], b[:2000], ".", ms=1.5, alpha=0.5)
    ax.set_title(f"{name}\nρ = {abs(rho) if abs(rho) < 0.005 else rho:.2f},  I = {info:.2f} bits", fontsize=9.5)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

In the parabola $x$ is symmetric around 0 and $y$ is fixed by $x$ up to the noise: a point at $x$ and one at $-x$
have the same $y$, so their products $x\,y$ cancel and $\rho = 0$. The mutual information, about 1 bit, sees
the dependence. The line is a Gaussian pair with $\rho = 0.8$: before cutting it has 0.737 bits (Part 4), and
8 bins keep 0.64 of them.

## Exercise 4: The bias of counting

Probabilities estimated from a finite number of samples fluctuate, and the fluctuations look like dependence.
Because the $I$ of any table of counts is $\ge 0$, they cannot cancel out: two signals independent of each
other give $I > 0$ on average. This is a **bias**. We measure it with **surrogates**, copies of region 5 whose
dependence on region 4 is removed on purpose:

- **shuffling** region 5 in time (`rng.permutation`) destroys every dependence on region 4, and also the slow
  time course of region 5;
- **rolling** region 5 by a random lag (a circular shift, `np.roll`) keeps its time course, because the shifted
  signal is the same signal started at another time. The BOLD signal stays correlated with itself for about 10
  volumes (7 s), so after a lag of at least 50 volumes (36 s) the shifted copy is no longer aligned with
  region 4. This approximates an independent signal with the same time course. The approximation assumes that
  the statistics do not change during the run: slow drifts or periodic components shared by the two regions
  would survive the shift.

Fill the two lists with 200 values each: enough to estimate their mean and their 95th percentile.

*Expected:* the shuffled values average about 0.005 bits, the value $(K_X - 1)(K_Y - 1)/(2 T \ln 2) = 0.0054$
bits predicted for $K = 4$ states and $T = 1200$ samples independent in time (Miller 1955). The circular shifts
average about 0.022 bits, four times as much: consecutive volumes are similar, so 1200 volumes hold only about
300 independent samples. 95% of the circular shifts stay below 0.05 bits, far below the 0.56 bits of the real
pair.

**Solution**

In [ ]:
rng = np.random.default_rng(0)
mi_shuffled, mi_shifted = [], []
for _ in range(200):
    mi_shuffled.append(mutual_information(X[4], rng.permutation(X[5])))
    mi_shifted.append(mutual_information(X[4], np.roll(X[5], rng.integers(50, T - 50))))

In [ ]:
bias_formula = (4 - 1) * (4 - 1) / (2 * T * np.log(2))
plt.figure(figsize=(6, 3.2))
bins = np.linspace(0, 0.08, 41)
plt.hist(mi_shuffled, bins=bins, alpha=0.8, label="shuffled")
plt.hist(mi_shifted, bins=bins, alpha=0.7, label="circular shift")
plt.axvline(bias_formula, color="#3D3D3D", ls=":", lw=1, label="formula, samples independent in time")
plt.xlabel("I between region 4 and a surrogate of region 5 (bits)")
plt.ylabel("surrogates")
plt.legend()
plt.show()
print(f"shuffled:       mean {np.mean(mi_shuffled):.4f} bits (formula {bias_formula:.4f})")
print(f"circular shift: mean {np.mean(mi_shifted):.4f} bits; 95th percentile {np.percentile(mi_shifted, 95):.4f}")
print(f"real pair:      {mutual_information(X[4], X[5]):.3f} bits")
lag1 = np.mean([np.corrcoef(x[:-1], x[1:])[0, 1] for x in bold])
lag10 = np.mean([np.corrcoef(x[:-10], x[10:])[0, 1] for x in bold])
print(f"autocorrelation of BOLD, mean over regions: {lag1:.2f} at 1 volume, {lag10:.2f} at 10 volumes")

---
# Part 4 — Mutual information and functional connectivity

The **functional connectivity** (FC) is the matrix of Pearson correlations between the BOLD signals of all pairs
of regions. The next cell computes the matrix of mutual information between the four-state signals of all
19,900 pairs of different regions ($200 \times 199 / 2$). The function `mi_matrix` does the same as
`mutual_information`, for all pairs at once: it counts the joint states of every pair with one matrix product
(the cell checks it on regions 4 and 5). The diagonal is left blank in the plot: there $I(X;X) = H(X) \approx 1.9$
bits, which would hide the colour scale of the pairs.

In [ ]:
def mi_matrix(X, n_states=4):
    """Mutual information (bits) between all pairs of rows of X (states 0 .. n_states-1)."""
    n, n_t = X.shape
    onehot = (X[:, None, :] == np.arange(n_states)[None, :, None]).reshape(n * n_states, n_t).astype(float)
    P = (onehot @ onehot.T).reshape(n, n_states, n, n_states).transpose(0, 2, 1, 3) / n_t   # p_ij(a, b)
    Pi, Pj = P.sum(axis=3), P.sum(axis=2)                                                  # marginals
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = np.where(P > 0, P * np.log2(P / (Pi[..., None] * Pj[..., None, :])), 0.0)
    return terms.sum(axis=(2, 3))


FC = np.corrcoef(bold)
MI = mi_matrix(X)
iu = np.triu_indices(N, 1)                          # the 19,900 pairs (upper triangle)
print(f"check: MI[4, 5] = {MI[4, 5]:.3f} bits, as in Exercise 3")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im = axes[0].imshow(FC, vmin=-1, vmax=1, cmap=FC_CMAP)
axes[0].set_title("FC (Pearson correlation)")
plt.colorbar(im, ax=axes[0], shrink=0.8)
im = axes[1].imshow(np.where(np.eye(N, dtype=bool), np.nan, MI), vmin=0, vmax=0.5, cmap=MI_CMAP)
axes[1].set_title("mutual information (bits)")
plt.colorbar(im, ax=axes[1], shrink=0.8)
for ax in axes:
    ax.set_xlabel("region")
    ax.set_xticks([0, 100, 199])
    ax.set_yticks([0, 100, 199])
plt.tight_layout()
plt.show()

The two matrices have the same blocks, the resting-state networks. The MI does not distinguish positive from
negative correlations: two regions that go up and down in opposition depend on each other as much as two that
go up and down together.

For two **jointly Gaussian** signals with correlation $\rho$, whose joint distribution is a two-dimensional
Gaussian, the mutual information follows from $\rho$ alone (the slides derive it, Part II):

$$I(X;Y) = -\tfrac{1}{2}\log_2\big(1 - \rho^2\big).$$

## Exercise 5: Mutual information of a Gaussian pair

Write `gaussian_mi`. It must accept an array of correlations (use `np.log2`), because the next cell passes it a
whole grid. That cell plots the MI of every pair of regions against its correlation, with two curves: your
formula, and the MI of a Gaussian pair cut at −1, 0, +1 like the data. The second curve is computed by
simulation: 200,000 samples of a Gaussian pair with correlation $\rho$, cut into four states and counted like
the data. With so many samples its bias is $9/(2 \cdot 200000 \ln 2) = 0.00003$ bits: the dashed curve is what
cutting alone does, without the bias of counting.

*Expected:* for $\rho = 0.80$, 0.737 bits. The dashed curve lies below yours, because cutting loses
information: at $\rho = 0.8$ it keeps 0.515 of the 0.737 bits. The points follow the dashed curve, on average
0.02 bits above it, about the bias of the circular shifts of Exercise 4. Weak pairs, whose information is
smaller than the bias, lie above both curves. A single pair can be further from the curve: regions 4 and 5
($ho = 0.804$) have 0.562 bits against 0.524 for a cut Gaussian pair with the same correlation, a difference
that mixes the bias, the random error of 14 minutes and any departure from a Gaussian pair.

**Solution**

In [ ]:
def gaussian_mi(rho):
    """Mutual information (bits) of two jointly Gaussian signals with correlation rho (a number or an array)."""
    return -0.5 * np.log2(1 - np.asarray(rho) ** 2)

In [ ]:
def cut_gaussian_mi(rho, n=200_000, seed=0):
    """MI (bits) of a Gaussian pair with correlation rho, cut at -1, 0, +1 like the data (by simulation)."""
    a, e = np.random.default_rng(seed).standard_normal((2, n))
    b = rho * a + np.sqrt(1 - rho**2) * e              # a Gaussian partner with correlation rho
    sa, sb = np.digitize(a, [-1, 0, 1]), np.digitize(b, [-1, 0, 1])
    joint = np.bincount(4 * sa + sb, minlength=16)     # counts of the 16 joint states
    return (entropy_bits(np.bincount(sa, minlength=4)) + entropy_bits(np.bincount(sb, minlength=4))
            - entropy_bits(joint))


print(f"rho = 0.80: Gaussian pair {gaussian_mi(0.8):.3f} bits, cut into four states {cut_gaussian_mi(0.8):.3f} bits")
rho_grid = np.linspace(-0.95, 0.95, 39)
mi_cut = np.array([cut_gaussian_mi(r) for r in rho_grid])

plt.figure(figsize=(6, 4))
plt.plot(FC[iu], MI[iu], ".", ms=1.5, alpha=0.25, color=PALETTE[0], label="pairs of regions")
plt.plot(rho_grid, gaussian_mi(rho_grid), color=PALETTE[7], lw=1.4, label="Gaussian pair")
plt.plot(rho_grid, mi_cut, color=PALETTE[1], lw=1.6, ls="--", label="Gaussian pair, four states")
plt.xlabel("correlation ρ")
plt.ylabel("mutual information (bits)")
plt.ylim(0, 1.2)
plt.legend(loc="upper center")
plt.show()
residual = MI[iu] - np.interp(FC[iu], rho_grid, mi_cut)
above = MI[iu] > gaussian_mi(FC[iu])
weak = np.abs(FC[iu]) < 0.3
print(f"distance from the dashed curve: mean {residual.mean():+.3f} bits, sd {residual.std():.3f} bits")
print(f"pairs above the black curve: {above[weak].mean():.0%} of those with |rho| < 0.3, "
      f"{above[~weak].mean():.0%} of the others")

**Question 2.** Where would the parabola of Part 3 appear on this plot? What does the plot say about a model
that reproduces only the FC, such as the whole-brain model of the next lesson?

**Answer.** At $\rho \approx 0$ with about 1 bit (with 8 bins; with our four states somewhat less), far above both curves:
a strong nonlinear dependence puts a pair above the information that its correlation allows a Gaussian pair.
The pairs of regions instead lie close to the dashed curve: on average 0.02 bits above it, about the bias of
counting, with a spread of 0.013 bits. At this resolution, four states and 14 minutes, the counted MI of the
pairs is compatible with that of Gaussian pairs with the same correlations. This does not show that the pairs
are Gaussian: two numbers that agree do not fix a whole distribution, and a nonlinear dependence smaller than
the bias would not be seen. For a model that reproduces only the FC it means that, at this resolution, it
also reproduces the pairwise MI we can measure.

---
# Part 5 — Three signals

Can three signals share information that no pair of them shows? Two systems of three binary signals $X$, $Y$,
$Z$ answer the question:

- **copies**: three copies of one fair coin, $X = Y = Z$;
- **XOR**: two independent fair coins $X, Y$, and $Z = X \oplus Y$ (exclusive or), which is 1 when $X \ne Y$ and
  0 when $X = Y$.

In the XOR system every pair is independent: knowing $X$ alone, $Z$ is still a fair coin, because $Y$ is. Yet
the three signals are not independent together: $Z$ is fixed by $X$ and $Y$, so
$p(x, y, z) \ne p(x)\,p(y)\,p(z)$. Pairwise independence is not independence. The information that a **pair**
of signals carries about a third is a mutual information like the others, with the pair treated as one signal
with four states:

$$I(Z; X, Y) = H(Z) + H(X, Y) - H(X, Y, Z).$$

In [ ]:
rng = np.random.default_rng(2)
coin1, coin2 = rng.integers(0, 2, (2, T))
copies = np.vstack([coin1, coin1, coin1])             # rows: X, Y, Z
xor = np.vstack([coin1, coin2, coin1 ^ coin2])

fig, axes = plt.subplots(1, 2, figsize=(10, 1.6))
for ax, S, title in [(axes[0], copies, "three copies"), (axes[1], xor, "XOR: $Z = X \\oplus Y$")]:
    ax.imshow(S[:, :60], aspect="auto", cmap=ListedColormap(["#F3F3F1", PALETTE[0]]), interpolation="nearest")
    ax.set_yticks([0, 1, 2], ["X", "Y", "Z"])
    ax.set_xlabel("time")
    ax.set_title(title)
plt.tight_layout()
plt.show()


def mi_with_pair(target, a, b):
    """Mutual information (bits) between the signal target and the pair (a, b), treated as one signal."""
    return joint_entropy(target) + joint_entropy(a, b) - joint_entropy(target, a, b)


for name, S in [("copies", copies), ("XOR", xor)]:
    x, y, z = S
    print(f"{name:7s} I(Z; X) = {mutual_information(z, x):.3f},  I(Z; Y) = {mutual_information(z, y):.3f},  "
          f"I(Z; X, Y) = {mi_with_pair(z, x, y):.3f} bits")

In both systems the pair $(X, Y)$ tells everything about $Z$: 1 bit. The difference is how. In the copies each
signal alone already gives the bit, and the second one repeats it: the information is **redundant**. In the XOR
neither signal alone gives anything, and the bit appears only when the two are known together: the information
is **synergistic**. Choose one signal, the **target** $Z$. The **O-information** of the three signals compares
what the other two tell about $Z$ separately with what they tell together:

$$\Omega = I(Z; X) + I(Z; Y) - I(Z; X, Y).$$

$\Omega > 0$ means that redundancy dominates (copies: $1 + 1 - 1 = +1$ bit), $\Omega < 0$ that synergy dominates
(XOR: $0 + 0 - 1 = -1$ bit). It is a balance: one triplet can hold both kinds, and the sign tells which one is
larger. Writing the three mutual informations with entropies gives
$\Omega = H(X) + H(Y) + H(Z) - H(X,Y) - H(X,Z) - H(Y,Z) + H(X,Y,Z)$, which does not change if we exchange the
signals: $\Omega$ does not depend on which signal we call the target. (For three signals this is the
O-information of Rosas et al. 2019; Optional B gives its form for more signals.)

## Exercise 6: The O-information

Write `o_information` with the definition above: `S` has three rows, and `target` is the row that plays the role
of $Z$. Use `mutual_information` and `mi_with_pair`.

*Expected:* copies $\Omega = +1.00$ bits, XOR $\Omega = -1.00$ bits, whichever row is the target.

**Solution**

In [ ]:
def o_information(S, target=2):
    """O-information (bits) of three signals, the rows of S: separately minus together."""
    z = S[target]
    x, y = [S[k] for k in range(3) if k != target]       # the other two signals
    return mutual_information(z, x) + mutual_information(z, y) - mi_with_pair(z, x, y)

In [ ]:
for name, S in [("copies", copies), ("XOR", xor)]:
    values = [o_information(S, target) for target in range(3)]
    print(f"{name:7s} Ω = " + ", ".join(f"{v:+.2f}" for v in values) + " bits (target X, Y, Z)")

## Triplets of brain regions

For three regions the four-state signals have 64 joint states, too many for 1200 slow volumes: counting would
bias $\Omega$ by about −0.05 bits, more than the typical value of $\Omega$ itself. We assume instead that the
three signals are **jointly Gaussian**. This is an assumption, not a result of Part 4: there the pairs were
only compatible with Gaussian pairs, and three signals can be Gaussian pair by pair without being jointly
Gaussian. Under this assumption nothing needs to be counted, and the formula of Part 4 extends to several
partners:

$$I(Z; X, Y) = -\tfrac12\log_2\big(1 - R^2\big),$$

where $R^2$ is the fraction of the variance of $Z$ explained by the best linear prediction from $X$ and $Y$
(least squares). For one partner $R^2 = \rho^2$, and the formula of Part 4 comes back. The normal equations of
least squares give $R^2$ from the three correlations:

$$R^2 = \frac{\rho_{xz}^2 + \rho_{yz}^2 - 2\,\rho_{xy}\,\rho_{xz}\,\rho_{yz}}{1 - \rho_{xy}^2}.$$

So, under this assumption, $\Omega$ of three regions follows from their three correlations: from the FC, but
only from three of its entries read together. It is an estimate of $\Omega$ built on the covariance: it sees
the synergy of the Gaussian part of the dependence, and a synergy beyond the correlations, like the XOR's,
would escape it. The next cell computes $\Omega$ for all 1,313,400 triplets of regions
($200 \cdot 199 \cdot 198 / 6$), in about a second, and prints the two extreme estimates.

In [ ]:
def gaussian_mi_R2(R2):
    """Mutual information (bits) between a Gaussian signal and its partners, from the variance explained."""
    return -0.5 * np.log2(1 - R2)


J, K = np.triu_indices(N, 1)
triplets, omega, odd = [], [], []
for i in range(N - 2):                               # triplets (i, j, k) with i < j < k; k is the target
    keep = J > i
    j, k = J[keep], K[keep]
    r_xy, r_xz, r_yz = FC[i, j], FC[i, k], FC[j, k]   # partners i and j, target k
    R2 = (r_xz**2 + r_yz**2 - 2 * r_xy * r_xz * r_yz) / (1 - r_xy**2)
    omega.append(gaussian_mi_R2(r_xz**2) + gaussian_mi_R2(r_yz**2) - gaussian_mi_R2(R2))
    odd.append(r_xy * r_xz * r_yz < 0)               # an odd number of negative correlations
    triplets.append(np.column_stack([np.full(len(j), i), j, k]))
omega, triplets, odd = np.concatenate(omega), np.concatenate(triplets), np.concatenate(odd)

plt.figure(figsize=(6, 3.2))
plt.hist(omega, bins=np.linspace(-0.8, 0.8, 161), color=PALETTE[0])
plt.axvline(0, color=PALETTE[7], lw=0.8)
plt.yscale("log")
plt.xlabel("Ω of a triplet of regions (bits)")
plt.ylabel("triplets")
plt.show()
print(f"{len(omega):,} triplets; Ω < 0 in {np.mean(omega < 0):.0%}; from {omega.min():+.2f} to {omega.max():+.2f} bits")
print(f"an odd number of negative correlations: {np.mean(odd):.1%} of the triplets, "
      f"Ω < 0 in {np.mean(omega[odd] < 0):.0%} of them")
for label, t in [("largest Ω", np.argmax(omega)), ("most negative Ω", np.argmin(omega))]:
    idx = triplets[t]
    r = FC[np.ix_(idx, idx)]
    print(f"\n{label}: Ω = {omega[t]:+.3f} bits")
    for i in idx:
        print("   ", region_name(i))
    print(f"    correlations: {r[0, 1]:+.2f} ({idx[0]}-{idx[1]}), {r[0, 2]:+.2f} ({idx[0]}-{idx[2]}), "
          f"{r[1, 2]:+.2f} ({idx[1]}-{idx[2]})")

Most triplets have $|\Omega|$ below 0.05 bits. Read the percentages as a description of this run, not as
a count of synergistic triplets in the brain:

- 47% is the fraction of negative estimates. A Gaussian triplet whose three correlations have a negative
  product, that is an odd number of negative correlations, always has $\Omega < 0$: that alone accounts for
  more than a third of the triplets. When $\Omega$ is close to 0 its sign can also change from one run to
  another.
- The two extremes are selected among 1.3 million estimates of the same run. Calling one triplet synergistic
  would need the same result in other runs or other participants.

The triplet with the largest $\Omega$ is a group of three strongly correlated regions: each partner repeats
the same shared fluctuation. The one with the most negative $\Omega$ is shaped like a sum: regions 26 and 142
do not correlate with each other, and both correlate with region 129. The next cell predicts region 129
linearly from each partner alone and from both together (least squares; no intercept is needed, because the
signals are z-scored), and turns the variance explained into bits.

In [ ]:
idx = triplets[np.argmin(omega)]
r = FC[np.ix_(idx, idx)]
weakest = min([(0, 1), (0, 2), (1, 2)], key=lambda p: abs(r[p]))          # the pair with no correlation
hub = idx[[k for k in range(3) if k not in weakest][0]]                   # the region linked to both
p1, p2 = idx[weakest[0]], idx[weakest[1]]
y = bold[hub]
info, unexplained = {}, {}
for name, cols in [(f"region {p1} alone", [p1]), (f"region {p2} alone", [p2]), ("both together", [p1, p2])]:
    A = np.column_stack([bold[c] for c in cols])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)                          # best linear prediction
    R2 = 1 - np.var(y - A @ coef) / np.var(y)
    info[name], unexplained[name] = gaussian_mi_R2(R2), 1 - R2
    print(f"region {hub} from {name}: R^2 = {R2:.2f}, unexplained {1 - R2:.2f}, I = {info[name]:.3f} bits")
alone = info[f"region {p1} alone"] + info[f"region {p2} alone"]
print(f"separately {alone:.3f} bits, together {info['both together']:.3f} bits: "
      f"Ω = {alone - info['both together']:+.3f} bits")
product = unexplained[f"region {p1} alone"] * unexplained[f"region {p2} alone"]
print(f"unexplained fractions: product of the two alone {product:.2f}, both together "
      f"{unexplained['both together']:.2f}")

**Question 3.** Regions 26 and 142, each on its own, tell 0.384 and 0.412 bits about region 129; together they
tell 1.553 bits, about twice the sum. Where does the extra information come from? Hint: the bits of the two
partners alone add up to $-\tfrac12\log_2$ of the *product* of their unexplained fractions. Would you notice this
triplet in the FC matrix?

**Answer.** Region 129 behaves like the sum of two signals that are independent of each other, one shared with region 26
and one with region 142: like $Z = X + Y$ plus noise, a Gaussian relative of the XOR. Each partner alone
predicts only its own part, and leaves 59% and 56% of the variance unexplained. Adding the two informations
corresponds to multiplying these fractions, $0.59 \times 0.56 = 0.33$; together the partners leave only 12%
unexplained, far less than 33%, so they tell more than the sum: $\Omega < 0$. In general a Gaussian triplet is
synergistic when the variance left by both partners is smaller than the product of the variances left by each,
redundant when it is larger, as when the two partners explain the same part. The FC shows two strong links and a
missing one, a pattern found in many triplets: under the Gaussian assumption the synergy is in the FC, but only
in three entries read together, never in one.

---
# Optional

**A. The choice of states.** Cut the signals in two (at 0) or into 8 equally populated bins
(`quantile_states(x, 8)`) and compute again $H(X_0)$, the MI of regions 4 and 5 and the mean of the circular
shifts. More states keep more detail; what happens to the bias?

**B. More than three signals.** The **total correlation** $TC = \sum_i H(X_i) - H(\mathbf X)$ (Watanabe 1960)
compares the whole with its parts, and the **dual total correlation**
$DTC = H(\mathbf X) - \sum_i H(X_i | \mathbf X^{-i})$ (Han 1978) is the entropy shared by at least two signals,
where $\mathbf X^{-i}$ are all the signals except $X_i$ and $H(X_i | \mathbf X^{-i}) = H(\mathbf X) -
H(\mathbf X^{-i})$. For any number of signals the O-information is $\Omega = TC - DTC$ (Rosas et al. 2019).
Write `total_correlation` and `dual_total_correlation` and check that, for three signals, $TC - DTC$ equals
your `o_information` for the copies (TC = 2, DTC = 1) and the XOR (TC = 1, DTC = 2).

**C. A phase-randomised copy of the data.** Multivariate phase randomisation (Prichard and Theiler 1994) adds
the same random phase to the Fourier transform of every region at each frequency. It keeps the power spectrum
of each region and the cross-spectra between regions, hence the correlations between regions. It does not
make the signals Gaussian by construction: a pure sinusoid stays a sinusoid. With many frequencies of
comparable power the copy is usually close to Gaussian, but check it, for example with a histogram of one
region. Compare the MI matrix of such a copy with that of the data:

```python
F = np.fft.rfft(bold, axis=1)
phase = np.random.default_rng(0).uniform(0, 2 * np.pi, F.shape[1])
phase[0] = phase[-1] = 0                         # keep the mean and the (real) Nyquist term
copy = np.fft.irfft(F * np.exp(1j * phase), n=T, axis=1)
copy = (copy - copy.mean(axis=1, keepdims=True)) / copy.std(axis=1, keepdims=True)
MI_copy = mi_matrix(np.digitize(copy, [-1, 0, 1]))
print(np.corrcoef(MI[iu], MI_copy[iu])[0, 1], np.mean(MI[iu] - MI_copy[iu]))
```

**D. Counting three regions.** Compute $\Omega$ of the triplet 26, 129, 142 by counting the four-state signals,
`o_information(X[[26, 142, 129]])`, and compare it with the Gaussian value. Then do the same with circularly
shifted copies of the three regions, which have no dependence at all: how large is the bias of counting?

**E. Time: transfer entropy.** What the present of $Y$ tells about the next value of $X$, beyond what the
present of $X$ already tells, is $T_{Y \to X} = H(X_{t+1}, X_t) + H(Y_t, X_t) - H(X_t) - H(X_{t+1}, Y_t, X_t)$
(Schreiber 2000, with a history of one step). Unlike $I$, it has a direction. Compute it both ways for regions
4 and 5 (`X[5][1:]` is $X_{t+1}$, `X[5][:-1]` is $X_t$) and compare with circular shifts.

---
# Summary

- Entropy is the average surprise, in bits. A BOLD signal cut into four states has 1.87 of the 2 possible
  bits. The chain rule $H(X,Y) = H(Y) + H(X|Y)$ connects joint and conditional entropy.
- Mutual information $I = H(X) + H(Y) - H(X,Y)$ is zero only for independent signals, whatever the shape of
  the dependence. Counting biases it upwards, more for slow signals: compare it with surrogates that keep the
  time course.
- For jointly Gaussian signals $I = -\tfrac12\log_2(1-\rho^2)$: the information depends on the correlation
  alone. At the resolution of four states and 14 minutes, the counted MI of pairs of resting-state BOLD
  signals is compatible with this formula.
- Three signals can share information that no pair shows. The O-information $\Omega$, what two signals tell
  about a third separately minus what they tell together, is positive when redundancy dominates and negative
  when synergy dominates; the XOR is invisible to every pair. Assuming that three regions are jointly
  Gaussian, their $\Omega$ follows from three entries of the FC.
- Next lesson: a whole-brain model fitted to the FC. Near its working point the model is linear with Gaussian
  fluctuations, so its correlations fix every quantity of today computed at zero lag.

**Reading:** Cover and Thomas (2006), ch. 2 (§2.1–2.8) and §8.1–8.6 for continuous signals; Timme and
Lapish (2018) for applications in neuroscience.

**References**
- Cover TM, Thomas JA (2006). *Elements of Information Theory*, 2nd ed. Wiley.
- Han TS (1978). Nonnegative entropy measures of multivariate symmetric correlations. *Inf Control* 36:133–156.
- Miller GA (1955). Note on the bias of information estimates. In: Quastler H (ed.), *Information Theory in Psychology*, Free Press, pp. 95–100.
- Prichard D, Theiler J (1994). Generating surrogate data for time series with several simultaneously measured variables. *Phys Rev Lett* 73:951–954.
- Rosas FE, Mediano PAM, Gastpar M, Jensen HJ (2019). Quantifying high-order interdependencies via multivariate extensions of the mutual information. *Phys Rev E* 100:032305.
- Schaefer A et al. (2018). Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity MRI. *Cereb Cortex* 28:3095–3114.
- Schreiber T (2000). Measuring information transfer. *Phys Rev Lett* 85:461–464.
- Shannon CE (1948). A mathematical theory of communication. *Bell Syst Tech J* 27:379–423, 623–656.
- Timme NM, Lapish C (2018). A tutorial for information theory in neuroscience. *eNeuro* 5:ENEURO.0052-18.2018.
- Van Essen DC et al. (2013). The WU-Minn Human Connectome Project: an overview. *NeuroImage* 80:62–79.
- Watanabe S (1960). Information theoretical analysis of multivariate correlation. *IBM J Res Dev* 4:66–82.
- Yeo BTT et al. (2011). The organization of the human cerebral cortex estimated by intrinsic functional connectivity. *J Neurophysiol* 106:1125–1165.